모듈 부르고 시스템 형성

In [ ]:
import pychrono as chrono
sys = chrono.ChSystemNSC()                                       # 예제1과 동일
sys.SetGravitationalAcceleration(chrono.ChVector3d(0, -9.81, 0)) # 예제1과 동일

# Bullet = 오픈소스 물리 엔진의 충돌 감지 알고리즘
sys.SetCollisionSystemType(chrono.ChCollisionSystem.Type_BULLET) # 충돌 시스템

print("=" * 55)
print("Lesson 02: 충돌과 바닥")
print("=" * 55)


접촉 재질 생성

In [ ]:
material = chrono.ChContactMaterialNSC()  # 부딪힐 때 재질 객체 
material.SetFriction(0.3)        # 마찰 계수 0.3
material.SetRestitution(0.7)     # 반발 계수 0.7 (0은 탄성 없음, 1은 거의 완전 탄성)

print(f"\n접촉 재질 생성:")
print(f"  마찰 계수:   0.3")
print(f"  반발 계수:   0.7 (꽤 잘 튕김)")

바닥 생성

In [ ]:
floor = chrono.ChBody()  # 바닥이라는 강체 생성
floor.SetMass(1)                                # 질량 (고정체라 무관하지만 필요)
floor.SetPos(chrono.ChVector3d(0, -1, 0))       # 위치: y=-1 (바닥면이 y=0이 되도록)
floor.SetFixed(True)                            # 바닥 공중에 고정
floor.EnableCollision(True)                     # 충돌 활성화

# 충돌 형태 추가: 가로 20m x 세로 2m x 깊이 20m 상자
floor.AddCollisionShape(chrono.ChCollisionShapeBox(material, 20, 2, 20))
sys.AddBody(floor)  # 시스템에 삽입

print(f"\n바닥 생성:")
print(f"  위치: y = {floor.GetPos().y} m")
print(f"  크기: 20m x 2m x 20m (상자)")
print(f"  고정: True (움직이지 않음)")

공 생성

In [ ]:
ball = chrono.ChBody()                          # 예제1과 동일
ball.SetMass(1.0)                               # 예제1과 동일
ball.SetPos(chrono.ChVector3d(0, 5, 0))         # 예제1과 동일
ball.SetFixed(False)                            # 예제1과 동일
ball.EnableCollision(True)                      # 충돌 활성화

# 충돌 형태 추가: 반지름 0.3m 구
ball.AddCollisionShape(chrono.ChCollisionShapeSphere(material, 0.3))

# 관성 모멘트 계산 (구의 공식: I = 2/5 * m * r²)
radius = 0.3
inertia = 2.0 / 5.0 * ball.GetMass() * radius**2
ball.SetInertiaXX(chrono.ChVector3d(inertia, inertia, inertia))

sys.AddBody(ball)  # 시스템에 공 추가

print(f"\n공 생성:")
print(f"  질량: {ball.GetMass()} kg")
print(f"  반지름: {radius} m")
print(f"  초기 높이: y = {ball.GetPos().y} m")
print(f"  충돌: 활성화됨")

시뮬 실행

In [ ]:
dt = 0.001       # 시간 간격: 1ms (충돌 정확도를 위해 작게)
end_time = 4.0   # 총 4초 시뮬레이션

print(f"\n{'시간(s)':>8}  {'높이(m)':>10}  {'속도(m/s)':>10}  상태")
print("─" * 50)

prev_vy = 0          # 이전 속도 (바운스 감지용)
bounce_count = 0     # 바운스 횟수

while sys.GetChTime() < end_time: # 예제 1과 동일
    sys.DoStepDynamics(dt)

    t = sys.GetChTime()      # 현재 시간
    y = ball.GetPos().y      # 높이
    vy = ball.GetPosDt().y   # y방향 속도

    if prev_vy < -0.5 and vy > 0:    # 바운스 카운트: 속도가 음→양으로 바뀌는 순간
        bounce_count += 1            # 작은 흔들림은 세지 않도록 여유롭게 0.5로 설정
        print(f"{t:8.3f}  {y:10.4f}  {vy:10.4f}  *** 바운스 #{bounce_count}! ***")

    # 0.2초마다 일반 출력
    elif abs(t % 0.2) < dt * 0.5 or abs(t % 0.2 - 0.2) < dt * 0.5:
        status = "낙하 중" if vy < -0.1 else ("상승 중" if vy > 0.1 else "정지")
        print(f"{t:8.3f}  {y:10.4f}  {vy:10.4f}  {status}")

    prev_vy = vy

결과 요약

In [ ]:
print(f"\n{'=' * 50}")
print(f"시뮬레이션 결과:")
print(f"  총 바운스 횟수: {bounce_count}")
print(f"  최종 높이:      {ball.GetPos().y:.4f} m")
print(f"  최종 속도:      {ball.GetPosDt().y:.4f} m/s")
print(f"  반발 계수:      0.7")
print(f"{'=' * 50}")

print(f"""
핵심 정리:
  1. 충돌하려면 두 물체 모두 EnableCollision(True)
  2. 충돌 형태(CollisionShape)를 반드시 추가해야 함
  3. 접촉 재질(ChContactMaterialNSC)로 마찰/반발 설정
  4. 바닥은 SetFixed(True)로 고정

다음 단계:
  → lesson_03에서 이 시뮬레이션을 Irrlicht 3D로 시각화합니다
""")
